# 03 Scoring And H3

This notebook documents the current H3 aggregation and scoring logic for the prototype.

The implemented local baseline uses:

- housing sale-price proxy from Anjuke
- nearest subway distance from the regression preprocessing step
- a `has_housing_sample` flag so missing housing data are not treated as cheap housing
- 2024 POI-based counts for schools, healthcare, groceries, convenience stores, parks, bus stops, and subway access
- bike / walk road availability proxies from the simplified Shanghai road network
- a 500 m processed grid layer as the intermediate scale before H3 aggregation
- mode-specific 15-minute radius proxy accessibility fields computed on the grid
- cached walk / bike road-network nearest-amenity access fields
- environmental layers from public APIs and remote sensing (`AQI` and Sentinel-2 `NDVI` are attached)

The current selected track is:

- **Track A — Healthy Lifestyle & Sport**


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
seed = json.loads((ROOT / "data" / "processed" / "shanghai_h3_seed.json").read_text(encoding="utf-8"))
df = pd.DataFrame(seed)
df.head()

## Grid Layer Preview

In [ ]:
grid_df = pd.read_json(ROOT / "data" / "processed" / "shanghai_grid_seed.json")
grid_df[[
    "grid_id",
    "proxy_access_walk",
    "proxy_access_bike",
    "proxy_access_transit",
    "proxy_access_car",
    "h3",
]].head()

## Key Score Fields

In [ ]:
score_cols = [c for c in df.columns if c.startswith("score_")]
score_cols

## Scoring Rationale

In [ ]:
manifest = json.loads((ROOT / "data" / "processed" / "project_manifest.json").read_text(encoding="utf-8"))
pd.Series(manifest["score_method"], name="description").to_frame()

## Track A Indicator Coverage

In [ ]:
pd.Series(manifest["track_indicator_status"], name="status").to_frame()

In [ ]:
df[
    [
        "score_baseline_walk",
        "score_track_walk",
        "score_composite_walk",
        "network_access_walk",
        "network_track_access_walk",
        "has_housing_sample",
        "avg_price_m2",
        "avg_subway_distance_m",
        "top_amenities",
    ]
].sort_values("score_composite_walk", ascending=False).head(15)

## Composite Score Distribution

In [ ]:
df[["score_composite_walk", "score_composite_bike", "score_composite_transit", "score_composite_car"]].describe().T

## Suggested Interpretation

The current prototype should be interpreted as a **screening surface** rather than a final
decision map. High-scoring hexes indicate where multiple supportive amenities, transit access, and
healthy-lifestyle signals overlap. Walk and bike now include local road-network access to key
amenities, while transit and car still rely on proxy surfaces instead of full timetable-aware or
routing-engine isochrones.


## Final-Step Upgrade Checklist

1. Replace POI density proxies with counts inside real 15-minute mode isochrones.
2. Replace stop-density transit proxies with GTFS-aware service coverage.
3. Replace sale-price proxy with true rent / affordability layers if Track C is chosen.
4. Deploy the public web app and record the final URL.
5. Document the weighting rationale with sensitivity tests.
